# Write Code to load data
Will later on be put into production as code in a .py file

Here it is to write and experiment with the code

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import socket as socket
import os as os
import sys as sys
import multiprocessing as mp
import warnings
import itertools as it

socket_name = socket.gethostname()
print(sys.version)
    
if socket_name.startswith("bionc") or socket_name.startswith("hpc"):
    print("Leipzig Cluster detected!")
    path = "/mnt/archgen/users/hringbauer/git/medieval/"   
else: 
    raise RuntimeWarning("Not compatible machine. Check!!")

sys.path.append("/mnt/archgen/users/hringbauer/git/EPIDEMIC/")
sys.path.append("/mnt/archgen/users/hringbauer/git/projectPCA/projectPCA/")
     
os.chdir(path)  # Set the right Path (in line with Atom default)
print(os.getcwd())
print(f"CPU Count: {mp.cpu_count()}")
print(sys.version_info)

from ancIBD.ibd_stats.funcs import new_columns, give_sub_df, remove_iids # To manipulate PMR tables
from python.eigenstrat_funcs import load_genos_autoeager, pw_mm_rate, pw_mm_rates_iids, calc_r

%config InlineBackend.print_figure_kwargs={'facecolor' : "w"}

3.12.3 (main, Jan 22 2026, 20:57:42) [GCC 13.3.0]
Leipzig Cluster detected!
/mnt/archgen/users/hringbauer/git/medieval
CPU Count: 128
sys.version_info(major=3, minor=12, micro=3, releaselevel='final', serial=0)


# 1) Code to load PLINK data

In [8]:
from bed_reader import open_bed

In [9]:
%%time

bed = open_bed(path_plink_files)

# Number of individuals and SNPs
print("Individuals:", bed.iid_count)
print("SNPs:", bed.sid_count)

Individuals: 335
SNPs: 5811803
CPU times: user 144 ms, sys: 61.2 ms, total: 205 ms
Wall time: 220 ms


In [102]:
class PlinkLoad():
    """Class that loads and postprocesses PLINK files.
    Same as Eigenstrat Superclass, but overwrites methods to load
    SNP and IID info as well as encoded Genotype Data"""
    base_path = "" # Path to the base before .bed
    bed_path = "" # Path to full bed file
    bed = None # 
    nind = 0 # How many IIDs in file
    nsnp = 0 # How many SNPs in file
    df_snp = []  # Dataframe with all SNPs
    df_ind = [] # Dataframe with all iids
    output = False # Whether to print detailed output.

    def __init__(self, base_path="", output=True):
        """Overwrite Concstructor:
        base_path: Data path without the .bed ending.
        output: Whether to print output."""
        self.output = output
        if len(base_path) > 0:
            self.base_path = base_path
            self.bed_path = self.base_path + ".bed"
        
        ### Get Size of Data Matrix and sanity check
        self.bed = open_bed(self.bed_path)
        self.nind = self.bed.iid_count
        self.nsnp = self.bed.sid_count

        ### Create the SNP and IND files
        self.df_snp = self.load_snp_df()   # Load the SNP DataFrame
        self.df_ind = self.load_ind_df()   # Load the Individual DataFrame

        ### Sanity Checks
        assert(len(self.df_snp) == self.nsnp)  # Sanity Check
        assert(len(self.df_ind) == self.nind)  # Sanity Check II

        if self.output:
            print(f"Loaded PLINK file set with {self.nind} Individuals and {self.nsnp} SNPs")

    def load_snp_df(self):
        """Load the SNP dataframe.
        Uses bed object"""
        if len(self.df_snp)==0:          
            # Create SNP metadata DataFrame
            df_snp = pd.DataFrame({
                "snp": self.bed.sid,                 # SNP ID
                "chr": self.bed.chromosome,          # Chromosome
                "map": self.bed.cm_position/100,         # Genetic map position (cM)
                "pos": self.bed.bp_position.astype("int"),         # Base pair position
                "ref": self.bed.allele_1,            # A1 allele (PLINK coding)
                "alt": self.bed.allele_2             # A2 allele
            })

        else:
            df_snp = self.df_snp
        return df_snp

    def load_ind_df(self):
        """Load the Individual dataframe.
        Uses bed object"""
        if len(self.df_ind)==0:
            df_ind = pd.DataFrame({"iid":self.bed.iid.astype("str"), 
                                   "sex":self.bed.sex.astype("str"), 
                                   "cls":""})
        else:
            df_ind = self.df_ind
        return df_ind

    def get_geno_all(self):
        """Load all genotypes from Eigenstrat File.
        Use self.nind for number of individuals.
        Return genotype matrix, with missing values set to missing_val"""
        gt = self.bed.read()
        gt = 2 - gt # To adjust for plink encoding
        return gt
    
    def get_geno_i(self, i):
        """Load Individual i"""
        geno_sub = self.bed.read(np.s_[i,:])
        geno_sub = 2 - geno_sub
        return geno_sub

    ##### 
    ### Can be deleted for inheritance
    def get_index_iid(self, iid):
        """Get Index of Individual iid"""
        # Detect the Individual
        found = np.where(self.df_ind["iid"] == iid)[0]
        if len(found)==0:
            raise RuntimeError(f"Individual {iid} not found in Eigenstrat!")
        else: 
            i = found[0]
        return i

    def get_geno_iid(self, iid):
        """Return Genotypes of Individual iid"""
        i = self.get_index_iid(iid)
        g = self.get_geno_i(i)
        return g

################
### Helper functions

def update_values(gt, x=[48,49,50,57], y=[2,1,0,9], copy=False):
    """"Update Values in numpy matrix gt. 
    x: List of original values.
    y: Updated values"""
    if copy:
        gt2 = gt.astype("float") # hard copy, necessary if values intersect

    else:
        gt2 = gt # only pointer
        
    for i,j in zip(x,y):
    #print(x,y)
        idx = gt == i
        gt2[idx] = j
    return gt2

### Alternative: Test the implemented version

In [2]:
### To test 
from loadEigenstrat import get_eigenstrat_object

### Test the class

In [3]:
%%time
es = get_eigenstrat_object(base_path="/mnt/archgen/users/hringbauer/git/EPIDEMIC/output/plink/bd_ptn_335", verbose=True,
                          mode="plink")

Loaded Genotype File set with 335 Individuals and 5811803 SNPs.


### Load Genotype of IID

In [4]:
%%time
g = es.get_geno_iid("PTN222")

CPU times: user 13.3 s, sys: 27.5 s, total: 40.8 s
Wall time: 22.9 s


In [5]:
np.mean(g)

np.float32(0.6342552)

In [6]:
g = es.get_geno_i(100)

In [7]:
np.mean(g)

np.float32(0.63174146)

# Area 51

In [109]:
%%time
g = bed.read(np.s_[0,:])

CPU times: user 14.3 s, sys: 1min 38s, total: 1min 52s
Wall time: 29.8 s


In [98]:
%%time
g1 = bed.read()

CPU times: user 17.7 s, sys: 26min 15s, total: 26min 33s
Wall time: 45.4 s


In [107]:
%%time
g1 = bed.read(dtype="int8")

CPU times: user 14.1 s, sys: 15min 50s, total: 16min 4s
Wall time: 30.8 s


In [108]:
%%time
2 - g1

CPU times: user 130 ms, sys: 7.25 s, total: 7.38 s
Wall time: 7.38 s


array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 1, 1, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 1, 1, 0],
       [0, 0, 0, ..., 2, 1, 0]], shape=(335, 5811803), dtype=int8)

In [99]:
a = np.array([1.0,np.nan])

In [101]:
2-a

array([ 1., nan])

In [105]:
print('test')

test
